# Confluence API Client

This notebook imports the reusable client from `confluence_client.py` and reads credentials from `.env`.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from confluence_client import ConfluenceAPIError, ConfluenceClient

client = ConfluenceClient.from_env()

In [ ]:
current_user = client.get_current_user()
{
    "display_name": current_user.get("displayName") or current_user.get("publicName"),
    "account_id": current_user.get("accountId"),
    "confluence_url": client.config.base_url,
}

In [ ]:
spaces = client.list_spaces(limit=5)
[
    {
        "key": space.get("key"),
        "name": space.get("name"),
        "type": space.get("type"),
    }
    for space in spaces.get("results", [])
]

In [ ]:
try:
    jira_roles_response = client.list_jira_application_roles()
    jira_apps = [
        {
            "key": app.get("key"),
            "name": app.get("name"),
            "user_count": app.get("userCount"),
            "groups": app.get("groups", []),
        }
        for app in jira_roles_response.get("items", [])
    ]
except ConfluenceAPIError as error:
    jira_apps = [
        {
            "error": str(error),
            "detail": error.response_body[:500],
        }
    ]

try:
    jira_response = client.list_jira_projects(max_results=50)
    jira_projects = [
        {
            "key": project.get("key"),
            "name": project.get("name"),
            "project_type": project.get("projectTypeKey"),
            "url": project.get("self"),
        }
        for project in jira_response.get("values", [])
    ]
except ConfluenceAPIError as error:
    jira_projects = [
        {
            "error": str(error),
            "detail": error.response_body[:500],
        }
    ]

confluence_base_url = client.config.base_url.rstrip("/")
if not confluence_base_url.endswith("/wiki"):
    confluence_base_url = f"{confluence_base_url}/wiki"

pages_response = client.list_pages(limit=20)
confluence_pages = []
for result in pages_response.get("results", []):
    page = result.get("content", result)
    links = page.get("_links", {})
    space = page.get("space", {})
    version = page.get("version", {})
    confluence_pages.append(
        {
            "title": page.get("title"),
            "space": space.get("name") or space.get("key"),
            "last_updated": version.get("when"),
            "url": f"{confluence_base_url}{links.get('webui', '')}",
        }
    )

{
    "jira_apps": jira_apps,
    "jira_projects": jira_projects,
    "confluence_pages": confluence_pages,
}

In [ ]:
# Replace this with any Confluence page URL you want to inspect.
page_url = confluence_pages[0]["url"] if confluence_pages else None

if page_url is None:
    first_page_response = client.list_pages(limit=1)
    first_page = first_page_response.get("results", [])[0]
    links = first_page.get("content", first_page).get("_links", {})
    confluence_base_url = client.config.base_url.rstrip("/")
    if not confluence_base_url.endswith("/wiki"):
        confluence_base_url = f"{confluence_base_url}/wiki"
    page_url = f"{confluence_base_url}{links.get('webui', '')}"

page_context = client.get_page_context_by_url(page_url)
{
    "title": page_context["title"],
    "space": page_context["space"],
    "url": page_context["url"],
    "jira_macros": page_context["jira_macros"],
    "text_preview": page_context["text"][:1000],
}

In [ ]:
page_jira_links = client.get_jira_links_from_page_url(page_url)
[
    {
        "key": jira_link["key"],
        "status": jira_link["status"],
        "status_category": jira_link["status_category"],
        "summary": jira_link["summary"],
        "assignee": jira_link["assignee"],
        "url": jira_link["url"],
        "error": jira_link["error"],
    }
    for jira_link in page_jira_links["jira_links"]
]